# House Stage 2: comparing error hierarchies H0–H3

| model | error components |
|---|---|
| H0, baseline | national + state + district (`02_monte_carlo.ipynb`) |
| H1, demographic | H0 + one national college-attainment coefficient per simulation × each district's loading (`src/house/simulation/factors.py`) |
| H2, regional | H0 + one shock per political region per simulation (`src/house/simulation/factors.py`) |
| H3, full | H0 + both (`src/house/simulation/factors.py`) |

Each hierarchy runs with both frozen Stage 1 configurations, 8 configurations in all, on the chronological backtest (2020 on 2018, 2022 on 2018–2020, 2024 on 2018–2022) and the 2026 forecast. Race handling, seeds and every other setting are the same across configurations, and only the error model differs. The factor variances use a zero prior of 3 pseudo-elections, fixed in advance; nothing below was tuned on these results.

The question is whether the factors improve the **joint** distribution of House outcomes (seats that swing together, especially competitive ones), not just district Brier scores. There are three test elections, so this is exploratory.

**Estimation** (`simulation.estimate_factor_sds`, `src/house/simulation/factors.py`, whose docstring has the details). Every component is fit on Stage 1's rolling out-of-sample residuals from the training elections only.

- **National:** H0's estimator, unchanged. The fixed 3-point prior SD counts as 2 pseudo-elections.
- **Demographic:** each year's within-state-year slope of the residual on $Z$ is one observation, net of its sampling variance. The coefficient variance is $\max(0, \sum_y(\hat\gamma_y^2 - se_y^2)) / (n_0 + K)$.
- **Regional:** each year's same-region minus different-region state-pair covariance, corrected for the bias that centering on the year mean causes (validated on synthetic data with unequal states and regions). The variance is $\max(0, \sum_y m_y) / (n_0 + K)$.
- **Shrinkage:** both factors use a zero prior with $n_0$ = 3 pseudo-elections. It was fixed in advance and is not tuned on this backtest.
- **Budget:** the factors take their variance out of H0's state and district levels, $\sigma_S^2 = \sigma_{S0}^2 - \sigma_R^2 - \sigma_\gamma^2 B$ and $\sigma_D^2 = \sigma_{D0}^2 - \sigma_\gamma^2 W$. An average district's error variance is therefore exactly H0's, and only the correlations change. A cap shrinks the factors themselves and is reported in `cap`.
- **Approximation:** the national estimator's yearly mean already contains the district-weighted average of that year's regional shocks. $\sigma_N$ is kept as in H0, so the simulated House-wide mean error can be slightly wider than in H0. `sim sd of mean error` reports it.

With only three backtest elections, every comparison here is exploratory.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # for house.experiments
import house.regression as hr
from house import simulation, stage1

In [ ]:
CONFIG = stage1.HOUSE_ENSEMBLE_AB_85_15
FALLBACK_CONFIG = stage1.HOUSE_MODEL_A
CALIBRATION_CONFIGS = {c.name: c for c in [stage1.HOUSE_MODEL_A, stage1.HOUSE_ENSEMBLE_AB_85_15]}
HOUSE_SEATS = 435

SEED = 42
N_SIMS = 40_000
NU = 5
NATIONAL_PRIOR_SD = 3.0
NATIONAL_PRIOR_ELECTIONS = 2
REGIONAL_PRIOR_ELECTIONS = 3
DEMOGRAPHIC_PRIOR_ELECTIONS = 3
COLLEGE_REFERENCE_SD = 10.0
CALIBRATION_YEARS = [2020, 2022, 2024]

In [ ]:
with warnings.catch_warnings(action="ignore"):  # loading warns about known optional-column gaps
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, 2006-2026
    targets = hr.load_targets()        # data/processed/house/targets.csv, 2006-2024
targets_raw = simulation.load_targets_raw()
candidates = simulation.load_candidates_2026()
races = simulation.historical_races(targets, targets_raw)  # lineups, treatments, observed winners and margins

stage1_runs = {name: simulation.stage1_inputs(config, predictors=predictors, targets=targets)
               for name, config in CALIBRATION_CONFIGS.items()}

REGIONS = simulation.load_regions(simulation.REGIONS_PATH)
LOADINGS = simulation.demographic_loadings(predictors, column="college", reference_sd=COLLEGE_REFERENCE_SD)
HIERARCHY_FACTORS = {"H0": (None, None), "H1": (None, LOADINGS), "H2": (REGIONS, None), "H3": (REGIONS, LOADINGS)}

## 1. Backtest: all 8 configurations

In [ ]:
calibration = {hierarchy: {} for hierarchy in HIERARCHY_FACTORS}
for hierarchy, (regions, loadings) in HIERARCHY_FACTORS.items():
    for name, run in stage1_runs.items():
        calibration[hierarchy][name] = simulation.backtest(
            run.oos,
            test_years=CALIBRATION_YEARS,
            races=races,
            predictions=run.predictions,
            fallback_predictions=stage1_runs[FALLBACK_CONFIG.name].predictions,
            nu=NU,
            n_sims=N_SIMS,
            seed=SEED,
            house_size=HOUSE_SEATS,
            national_prior_sd=NATIONAL_PRIOR_SD,
            national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
            regions=regions,
            loadings=loadings,
            regional_prior_elections=REGIONAL_PRIOR_ELECTIONS,
            demographic_prior_elections=DEMOGRAPHIC_PRIOR_ELECTIONS,
        )
        print(hierarchy, simulation.check_backtest(
            calibration[hierarchy][name], run, races, house_size=HOUSE_SEATS, n_sims=N_SIMS,
            national_prior_sd=NATIONAL_PRIOR_SD, national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
            regions=regions, loadings=loadings, regional_prior_elections=REGIONAL_PRIOR_ELECTIONS,
            demographic_prior_elections=DEMOGRAPHIC_PRIOR_ELECTIONS))
        for year, result in calibration[hierarchy][name].years.items():
            if hierarchy != "H0":  # H0 is the budget
                simulation.check_variance_budget(result.error_sd, result.error_details,
                                                 calibration["H0"][name].years[year].error_sd)
print("variance budget checks passed for every hierarchy, configuration and test year")

In [ ]:
# H0 here is 02_monte_carlo.ipynb's backtest exactly: the same estimator, draws and scores
h0_reference = simulation.backtest(stage1_runs[CONFIG.name].oos, test_years=CALIBRATION_YEARS, races=races,
                                   predictions=stage1_runs[CONFIG.name].predictions,
                                   fallback_predictions=stage1_runs[FALLBACK_CONFIG.name].predictions, nu=NU,
                                   n_sims=N_SIMS, seed=SEED, house_size=HOUSE_SEATS,
                                   national_prior_sd=NATIONAL_PRIOR_SD,
                                   national_prior_elections=NATIONAL_PRIOR_ELECTIONS)
for year in CALIBRATION_YEARS:
    print(simulation.check_same_draws(h0_reference.years[year], calibration["H0"][CONFIG.name].years[year]))

### Calibration by election

Brier scores (all resolved races, contested D-vs-R, ex-ante competitive), 80% and 95% margin coverage with mean widths, and seat medians, intervals and percentiles over the matched panel.

In [ ]:
table = simulation.stack(calibration, "table", outer="hierarchy")
widths = simulation.stack(calibration, "diagnostics", outer="hierarchy")[["80% mean width", "95% mean width"]]
comparison = table.drop(columns=["stage 2 training", "matched panel", "unresolved"]).join(widths)
simulation.styled(comparison.sort_index(level=["config", "test year", "hierarchy"]))

In [ ]:
# Race-level metrics pooled over 2020-2024 (seat counts are not pooled)
simulation.styled(pd.DataFrame({(h, name): cal.pooled() for h, cals in calibration.items()
                                for name, cal in cals.items()}).T.rename_axis(["hierarchy", "config"]))

### Component scales

The fitted SD of every component per test year, the allocated SDs, and the simulated SD of the House-wide mean error. That last column shows how far the regional shock overlaps the national one; $\sigma_N$ itself is the same in every hierarchy.

In [ ]:
diagnostics = simulation.stack(calibration, "diagnostics", outer="hierarchy")
simulation.styled(diagnostics.filter(regex=" sd$|sim sd").sort_index(level=["config", "test year", "hierarchy"]))

In [ ]:
# Any binding cap, and the final allocated variance components (H3 especially)
caps = pd.concat({(h, name, year): result.error_details[["regularized sd", "sd", "district variance", "cap"]]
                  for h, cals in calibration.items() for name, cal in cals.items()
                  for year, result in cal.years.items() if h != "H0"}, names=["hierarchy", "config", "test year"])
print("binding caps:", int((caps["cap"] != "").sum()))
caps.xs("house_ensemble_ab_85_15", level="config").round(3)

## 2. Joint distribution

`simulation.group_seats` scores each group's seat count: whether the actual count falls inside the group's 80% and 95% intervals, and how wide they are. A missing correlation makes a group's seats too narrow, so they miss too often. Region scoring favors H2 by construction and college-tercile scoring favors H1; the competitive split (ex-ante competitive D-vs-R races vs the other simulated races) is neutral. Each grouping gives a few dozen counts at most.

In [ ]:
region_groups = races[hr.KEY].assign(region=races["state"].map(REGIONS))
college_groups = simulation.college_terciles(LOADINGS)
group_tables = {
    "region": simulation.stack(calibration, lambda cal: simulation.group_seats(cal, "region", region_groups),
                               outer="hierarchy"),
    "college tercile": simulation.stack(calibration, lambda cal: simulation.group_seats(cal, "college_tercile",
                                                                                        college_groups),
                                        outer="hierarchy"),
    "competitive": simulation.stack(calibration, lambda cal: simulation.group_seats(cal, "competitive_ex_ante"),
                                    outer="hierarchy"),
}
group_summary = pd.concat({grouping: table.groupby(["config", "hierarchy"])[
    ["inside 80%", "inside 95%", "80% seat width", "95% seat width"]].mean()
    for grouping, table in group_tables.items()}, names=["grouping"])
group_summary.round(3)

In [ ]:
# Competitive seats: each election's simulated interval and the actual count, by hierarchy
competitive = group_tables["competitive"].xs(True, level="competitive_ex_ante")
simulation.styled(competitive.sort_index(level=["config", "test year", "hierarchy"]))

## 3. Seat distributions, H0–H3

In [ ]:
PLOT_YEAR = 2020        # the miss: actual Democratic seats at about the 14th percentile under H0
PLOT_CONFIG = stage1.HOUSE_ENSEMBLE_AB_85_15

fig, axes = simulation.plot_backtest_seat_distributions({h: calibration[h][PLOT_CONFIG.name] for h in calibration},
                                                        years=[PLOT_YEAR])

In [ ]:
fig, ax = simulation.plot_seat_errors({h: calibration[h][PLOT_CONFIG.name] for h in calibration})

## 4. 2026 under each hierarchy

The production forecast (the ensemble, Model A fallback, every race of the 2026 registry) with each error model fit on 2018–2024 and the same draws seed.

In [ ]:
FORECAST_YEAR = 2026
forecast = simulation.stage1_forecast(CONFIG, predictors=predictors, targets=targets)
fallback_forecast = simulation.stage1_forecast(FALLBACK_CONFIG, predictors=predictors, targets=targets)
point_2026 = simulation.attach_predictions(simulation.races_2026(candidates), forecast.predictions,
                                           primary_source=CONFIG.name, fallback=fallback_forecast.predictions)
oos = stage1_runs[CONFIG.name].oos
H0_SD_2026, _ = simulation.estimate_error_sds(oos, national_prior_sd=NATIONAL_PRIOR_SD,
                                              national_prior_elections=NATIONAL_PRIOR_ELECTIONS)

draws_2026, sds_2026 = {}, {}
for hierarchy, (regions, loadings) in HIERARCHY_FACTORS.items():
    sds_2026[hierarchy], details = simulation.estimate_factor_sds(
        oos,
        regions=regions,
        loadings=loadings,
        national_prior_sd=NATIONAL_PRIOR_SD,
        national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
        regional_prior_elections=REGIONAL_PRIOR_ELECTIONS,
        demographic_prior_elections=DEMOGRAPHIC_PRIOR_ELECTIONS,
    )
    simulation.check_variance_budget(sds_2026[hierarchy], details, H0_SD_2026)
    point = simulation.attach_factors(point_2026, regions=regions, loadings=loadings)
    scales = {c: simulation.t_scale(sd, NU) for c, sd in sds_2026[hierarchy].items()}
    draws_2026[hierarchy] = simulation.simulate_races(point, scales, NU, N_SIMS,
                                                      np.random.default_rng([SEED, FORECAST_YEAR]))
    print(hierarchy, simulation.check_forecast(simulation.forecast_table(draws_2026[hierarchy], candidates),
                                               draws_2026[hierarchy], candidates, house_size=HOUSE_SEATS))
display(pd.DataFrame(sds_2026).T.round(3))
simulation.styled(pd.concat({h: simulation.seat_table(d, {"official": None}) for h, d in draws_2026.items()},
                            names=["hierarchy"]).xs("D", level="party"))

In [ ]:
fig, axes = plt.subplots(len(draws_2026), 1, figsize=(9, 3.4 * len(draws_2026)), layout="constrained", sharex=True)
for ax, (hierarchy, draws) in zip(axes, draws_2026.items()):
    simulation.plot_seat_distribution(draws.seats(simulation.DEM), year=FORECAST_YEAR, actual_seats=None,
                                      panel_size=HOUSE_SEATS, house_size=HOUSE_SEATS,
                                      title=f"2026 · {hierarchy} · {CONFIG.name}", ax=ax, compact=True)

## 5. Interpretation (exploratory: three test elections, nothing tuned)

- **The factors are small once fitted and shrunk.** The regional SD is 0.8–1.4 points in the backtest windows (1.44 for 2026; 1.8–1.9 before shrinkage). The demographic coefficient SD is 0–0.34 points per 10 points of college share: 0 for 2020, because 2018's slope was within its own sampling error. Together they move about 2 of an average district's roughly 42 points² of sub-national error variance out of the state level. No cap binds.
- **District metrics don't move.** Contested and competitive Brier agree across H0–H3 to the third decimal, margin coverage moves by at most one race, and seat intervals move by at most one seat.
- **The joint distribution barely moves either.** Group-seat coverage (regions, college terciles, competitive seats) is identical across the four hierarchies for both Stage 1 configurations. The 2020 seat miss (actual at about the 14th percentile) is unchanged: it is a level miss shared by every race, which correlations between subsets cannot fix.
- **National overlap.** H2/H3 raise the simulated SD of the House-wide mean error by 0.01–0.04 points over H0's 2.64–2.93. That is the expected small double count with σ_N, and it is negligible.
- **2026.** The median is 231 D seats under every hierarchy, with the same 80% and 95% intervals. P(218+) is 94.7% (H0), 94.6% (H1), 94.2% (H2) and 94.3% (H3).

**Reading.** With strong shrinkage and the variance budget held fixed, neither factor changes the House outcome distribution in a way this backtest could detect. The data give no evidence that H1–H3 improve on H0, and no evidence they hurt it. Any case for a factor would rest on priors about structure, not on these three elections.